# Parallel transport and DEM slope

`transport_model(parallel=True)` maps one shape per worker through
`map_shapes()`. `slope_class_europe_emep()` demonstrates the EEA slope buckets
used by the EMEP lookup.


In [ ]:
import operator
from pathlib import Path

from gtfs2emis_py import transport_model
from gtfs2emis_py.gtfs.parallel import ShapeTask, map_shapes

DATA = Path.cwd() / "tests/data"
model = transport_model(DATA / "synthetic_edge_cases.zip", parallel=True, ncores=2)
print(model.manifest.parameters["parallel"], model.manifest.parameters["ncores"])
print("shapes:", model.manifest.counts["shapes"])

tasks = [ShapeTask("A", "alpha"), ShapeTask("B", "beta")]
outcome = map_shapes(operator.attrgetter("shape_id"), tasks, n_jobs=2)
print(outcome.results)

In [ ]:
import geopandas as gpd
from shapely.geometry import LineString

from gtfs2emis_py.gtfs.slope import classify_slope, slope_class_europe_emep

table = gpd.GeoDataFrame(
    {"dist_km": [0.2, 0.2]},
    geometry=[
        LineString([(-49.270, -25.430), (-49.269, -25.429)]),
        LineString([(-49.269, -25.429), (-49.268, -25.428)]),
    ],
    crs="EPSG:4326",
)
classified = slope_class_europe_emep(table, DATA / "bra_cur-srtm.tif", keep=True)
print(classify_slope([-0.08, -0.01, 0.0, 0.07, 0.0701]).tolist())
classified[["dist_km", "dh", "slope", "slope_class"]]

`map_shapes()` preserves input order and records failures without
dropping successful results. Slope values above `0.07` map to `-0.06`; exactly
`0.07` stays in `0.06`, and empty/out-of-raster segments are flat with a warning.
